# MAO-RAG Offline Colab Runner

Notebook n?y ???c thi?t k? ?? ch?y tr?n Google Colab GPU T4 sau khi clone repo `sinh2206/vn_stock_mao_arag.git`.

M?c ti?u c?a `Run all`:

1. Clone ho?c c?p nh?t repo.
2. C?i m?i tr??ng.
3. Chu?n b? `.env` cho Colab.
4. Chu?n b? data t? `data/chunks`, `storage_rag`, ho?c demo fallback.
5. T?i model c?n thi?t.
6. Ch?y sanity test pipeline RAG.
7. Dry-run PPO cho Qwen planner.

Ghi ch? t?i nguy?n: Qwen2.5-7B-Instruct 4-bit c? th? ch?y tr?n T4 trong nhi?u tr??ng h?p. MiniMax-M2.1 r?t n?ng, m?c ??nh t?t tr?n T4 ?? notebook kh?ng OOM. Executor v?n ch?y extractive QA b?ng heuristic fallback.


In [ ]:
#@title 1. Runtime configuration
REPO_URL = "https://github.com/sinh2206/vn_stock_mao_arag.git"
PROJECT_DIR = "/content/vn_stock_mao_arag"

# T4-safe defaults. Change only when you know the runtime has enough VRAM/RAM.
USE_QWEN_PLANNER = True
USE_MINIMAX_EXECUTOR = False
USE_RERANKER = False
LOAD_IN_4BIT = True

# Model downloads. Qwen is large; set DOWNLOAD_QWEN=False to use heuristic planner only.
DOWNLOAD_EMBEDDER = True
DOWNLOAD_CROSS_ENCODER = False
DOWNLOAD_QWEN = USE_QWEN_PLANNER
DOWNLOAD_MINIMAX = USE_MINIMAX_EXECUTOR

# Data/index behavior.
PROCESS_CAFEF_IF_PRESENT = True
MAX_STORAGE_DOCS_FOR_FALLBACK = 5000
RAG_RETRIEVAL_MODE = "sparse_only"  # sparse_only is fastest and always works in Colab.

# Optional heavy RL step. Dry-run always runs; actual PPO is off by default.
RUN_PPO_TRAINING = False

SAMPLE_QUESTION = "T?m t?t c?c tin CafeF li?n quan ??n FPT v? cho bi?t sentiment ch?nh n?u c?."


In [ ]:
#@title 2. Clone repository and install dependencies
import os, subprocess, sys, textwrap
from pathlib import Path


def run(cmd, cwd=None, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

if not Path(PROJECT_DIR).exists():
    run(f"git clone {REPO_URL} {PROJECT_DIR}")
else:
    run("git pull --ff-only", cwd=PROJECT_DIR, check=False)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

# Colab usually already has torch. Requirements install the rest.
run("python -m pip install -q --upgrade pip")
run("python -m pip install -q -r requirements.txt")

# Basic runtime diagnostics.
run("python - <<'PY'
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY")


In [ ]:
#@title 3. Write Colab .env
from pathlib import Path

env = """# Auto-generated by main.ipynb for Colab.
ENABLE_LOCAL_PLANNER={enable_planner}
ENABLE_LOCAL_EXECUTOR={enable_executor}
ENABLE_RERANKER={enable_reranker}
LOCAL_FILES_ONLY=true
LOAD_IN_4BIT={load_4bit}

QWEN_MODEL_NAME=models/qwen
MINIMAX_MODEL_NAME=models/minimax
EMBEDDING_MODEL_NAME=models/embedder
RERANKER_MODEL_NAME=models/cross_encoder

RAG_RETRIEVAL_MODE={retrieval_mode}
RAG_DENSE_WEIGHT=0.65
RAG_SPARSE_WEIGHT=0.35
RAG_TOP_K=10
RAG_EXECUTOR_TOP_K=5
RAG_DOCUMENT_PATH=data/chunks/colab_chunks.json
RAG_INDEX_PATH=data/index
""".format(
    enable_planner=str(USE_QWEN_PLANNER).lower(),
    enable_executor=str(USE_MINIMAX_EXECUTOR).lower(),
    enable_reranker=str(USE_RERANKER).lower(),
    load_4bit=str(LOAD_IN_4BIT).lower(),
    retrieval_mode=RAG_RETRIEVAL_MODE,
)
Path(".env").write_text(env, encoding="utf-8")
print(Path(".env").read_text(encoding="utf-8"))


In [ ]:
#@title 4. Prepare data for Colab
import json, os, subprocess, sys
from pathlib import Path

root = Path.cwd()
chunks_target = Path("data/chunks/colab_chunks.json")
chunks_target.parent.mkdir(parents=True, exist_ok=True)
Path("data/documents").mkdir(parents=True, exist_ok=True)
Path("data/training").mkdir(parents=True, exist_ok=True)
Path("data/metadata").mkdir(parents=True, exist_ok=True)

raw_cafef = Path("cafef_news-20260211T204544Z-1-001")
processed_cafef_chunks = Path("data/chunks/cafef_news_chunks.json")
repo_chunks = Path("data/chunks/chunks.json")

if PROCESS_CAFEF_IF_PRESENT and raw_cafef.exists():
    run("python scripts/process_cafef_news.py --input_dir cafef_news-20260211T204544Z-1-001")
    if processed_cafef_chunks.exists():
        chunks_target.write_text(processed_cafef_chunks.read_text(encoding="utf-8"), encoding="utf-8")
        print("Using processed CafeF chunks:", processed_cafef_chunks)
elif processed_cafef_chunks.exists():
    chunks_target.write_text(processed_cafef_chunks.read_text(encoding="utf-8"), encoding="utf-8")
    print("Using existing CafeF chunks:", processed_cafef_chunks)
elif repo_chunks.exists():
    chunks_target.write_text(repo_chunks.read_text(encoding="utf-8"), encoding="utf-8")
    print("Using repo chunks:", repo_chunks)
else:
    # Fallback: extract TextNode-like records from storage_rag docstores.
    docs = []
    storage = Path("storage_rag")
    if storage.exists():
        for docstore in storage.glob("*/docstore.json"):
            ticker = docstore.parent.name
            try:
                payload = json.loads(docstore.read_text(encoding="utf-8"))
                data = payload.get("docstore/data", {})
                for node_id, node in data.items():
                    item = node.get("__data__", {})
                    text = item.get("text", "")
                    if text and len(text.strip()) > 40:
                        docs.append({
                            "id": f"{ticker}_{node_id}",
                            "text": text,
                            "metadata": {"ticker": ticker, **(item.get("metadata") or {})},
                        })
                    if len(docs) >= MAX_STORAGE_DOCS_FOR_FALLBACK:
                        break
            except Exception as exc:
                print("skip", docstore, exc)
            if len(docs) >= MAX_STORAGE_DOCS_FOR_FALLBACK:
                break
    if not docs:
        docs = [
            {"id": "demo_fpt", "text": "Ticker: FPT
Title: FPT ho?n th?nh t?ng v?n ?i?u l? l?n h?n 17 ngh?n t? ??ng.
Sentiment: positive=0.72, neutral=0.20, negative=0.08", "metadata": {"ticker": "FPT"}},
            {"id": "demo_hpg", "text": "Ticker: HPG
Title: HPG ghi nh?n nhi?u tin li?n quan ??n th?p v? th? tr??ng b?t ??ng s?n.
Sentiment: positive=0.41, neutral=0.45, negative=0.14", "metadata": {"ticker": "HPG"}},
            {"id": "demo_vcb", "text": "Ticker: VCB
Title: VCB c? nhi?u tin li?n quan nh?m ng?n h?ng v? th? tr??ng chung.
Sentiment: positive=0.36, neutral=0.50, negative=0.14", "metadata": {"ticker": "VCB"}},
        ]
    chunks_target.write_text(json.dumps(docs, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Using fallback chunks from storage_rag/demo:", len(docs))

chunks = json.loads(chunks_target.read_text(encoding="utf-8"))
print("Prepared chunks:", len(chunks), "->", chunks_target)
print(json.dumps(chunks[0], ensure_ascii=False)[:800])


In [ ]:
#@title 5. Download required models
from pathlib import Path

model_flags = []
if DOWNLOAD_EMBEDDER:
    model_flags.append("embedder")
if DOWNLOAD_CROSS_ENCODER:
    model_flags.append("cross_encoder")
if DOWNLOAD_QWEN:
    model_flags.append("qwen")
if DOWNLOAD_MINIMAX:
    model_flags.append("minimax")

if model_flags:
    run("python scripts/download_models.py --only " + " ".join(model_flags))
else:
    print("No model download requested.")

for folder in ["models/embedder", "models/cross_encoder", "models/qwen", "models/minimax"]:
    p = Path(folder)
    print(folder, "exists=", p.exists(), "files=", len(list(p.rglob('*'))) if p.exists() else 0)


In [ ]:
#@title 6. Run pipeline sanity check
import json, os
from pathlib import Path

from agents.executor_agent import ExecutorAgent
from agents.planner_agent import PlannerAgent
from agents.reranker_agent import RerankerAgent
from agents.retriever_agent import RetrieverAgent
from core.orchestrator import Orchestrator, OrchestratorConfig
from rag_engine.retriever import HybridRetriever, HybridRetrieverConfig
from rag_engine.schema import Document

chunks = json.loads(Path("data/chunks/colab_chunks.json").read_text(encoding="utf-8"))
documents = [Document.from_any(item, index=i) for i, item in enumerate(chunks)]
print("Loaded documents:", len(documents))

retriever = HybridRetriever(
    documents=documents,
    config=HybridRetrieverConfig(mode=RAG_RETRIEVAL_MODE, output_top_k=10),
)

planner = PlannerAgent(
    model_name="models/qwen",
    enable_llm=USE_QWEN_PLANNER and Path("models/qwen/config.json").exists(),
    local_files_only=True,
    load_in_4bit=LOAD_IN_4BIT,
    max_new_tokens=512,
)
executor = ExecutorAgent(
    model_name="models/minimax",
    enable_model=USE_MINIMAX_EXECUTOR and Path("models/minimax/config.json").exists(),
    local_files_only=True,
    load_in_4bit=LOAD_IN_4BIT,
)
reranker = RerankerAgent(
    model_name="models/cross_encoder",
    enable_model=USE_RERANKER and Path("models/cross_encoder/config.json").exists(),
)

orchestrator = Orchestrator(
    planner_agent=planner,
    retriever_agent=RetrieverAgent(retriever=retriever),
    reranker_agent=reranker,
    executor_agent=executor,
    config=OrchestratorConfig(retriever_top_k=10, executor_top_k=5),
)

result = orchestrator.run(SAMPLE_QUESTION)
print("QUESTION:", SAMPLE_QUESTION)
print("ANSWER:", result["answer"])
print("PLAN:")
print(json.dumps(result["plan"], ensure_ascii=False, indent=2)[:2000])
print("SUB ANSWERS:", len(result["sub_answers"]))


In [ ]:
#@title 7. PPO readiness check for Qwen planner
run("python scripts/train_qwen_ppo.py --dry_run")

if RUN_PPO_TRAINING:
    run("python scripts/train_qwen_ppo.py --model_path models/qwen --dataset data/training/cafef_planner_workflows.jsonl --output_dir models/qwen_ppo --max_steps 10")
else:
    print("PPO actual training skipped. Set RUN_PPO_TRAINING=True in cell 1 to train.")


In [ ]:
#@title 8. Optional: launch Streamlit UI in Colab
LAUNCH_STREAMLIT_UI = False

if LAUNCH_STREAMLIT_UI:
    import subprocess, sys, time
    proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "main.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
    time.sleep(5)
    print("Streamlit process started. Expose port 8501 with your preferred Colab tunnel service.")
else:
    print("UI launch skipped. To use UI locally: streamlit run main.py")
